# 04 Model: baselines, rank by roof yield and by neighbour PV share

Drafted with an AI assistant, see docs/ai_log.md.

Input: `data/processed/model_table.parquet`.
Output: `data/processed/predictions.parquet` (`egid, cutoff, split, model, score`).

Two models, nothing is fitted:
- `roof_yield`: yearly electricity yield of the suitable roof surfaces.
- `nbr_share`: share of PV buildings among the neighbours within 250 m (`nbr_pv_share_250`; the radius was fixed before any result, see docs/decisions.md).

Each model adds one frame to `frames`; later models do the same.

In [ ]:
import helpers
import pandas as pd

In [ ]:
model_table = pd.read_parquet(helpers.PROCESSED / "model_table.parquet")
keys = model_table[["egid", "cutoff", "split"]]
frames = []

# Baseline 1: roof yield
frames.append(keys.assign(model="roof_yield", score=model_table["yield_kwh"]))

# Baseline 2: neighbour PV share at 250 m
frames.append(keys.assign(model="nbr_share", score=model_table["nbr_pv_share_250"]))

In [ ]:
# One table, sorted deterministically; scores as float64 rounded to 6 decimals
predictions = pd.concat(frames, ignore_index=True)
predictions["score"] = predictions["score"].astype("float64").round(6)
predictions = predictions.sort_values(["cutoff", "model", "egid"]).reset_index(drop=True)
print(predictions["score"].isna().sum(), "missing scores")
print(predictions.groupby(["model", "split"]).size())
predictions.to_parquet(helpers.PROCESSED / "predictions.parquet", index=False)